# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm
import random

# In-/ Output Structure & Class-Mapping

In [2]:
DATASETS_TO_PROCESS = ("turning", "cnc_machining")

DATASET_CONFIGS = {
    "turning": {
        "data_root": Path("../data/01_windowed_labeled_2,5s"),
        "output_root": Path("../data/02_spectrograms_150x100px_dataset"),
        "manifest_path": Path("../reports/manifests/turning_split_seed42.csv"),
        "sample_rate_hz": 50000,
    },
    "cnc_machining": {
        "data_root": Path("../data/01_windowed_labeled_cnc_machining"),
        "output_root": Path("../data/02_spectrograms_150x100px_cnc_machining_dataset"),
        "manifest_path": Path("../reports/manifests/cnc_machining_split_seed42.csv"),
        "sample_rate_hz": 2000,
    },
}

REPO_ROOT = Path("..").resolve()

CLASS_MAP = {
    "chatter": "chatter",
    "no_chatter": "no_chatter" 
}

SEED = 42
NPERSEG = 512
MAX_FREQ_CUT = 5000
DB_PERCENTILES = (1, 99)
DB_RANGE_SAMPLE_LIMIT = 1000
EPS = 1e-12

TARGET_IMG_SIZE = (150, 100)


# Spectrogram configuration
- NPERSEG: controls frequency resolution of spectrogram
larger values → better frequency resolution, lower time resolution
- MAX_FREQ_CUT: limits analysis to relevant chatter frequency range
- DB_PERCENTILES: spectrogram is converted to decibel scale and robustly scaled per dataset from training data
- DB_RANGE_SAMPLE_LIMIT: maximum number of training windows used to estimate the dataset-level dB range
- TARGET_IMG_SIZE: image size of three color spectrogram

In [3]:
for config in DATASET_CONFIGS.values():
    config["output_root"].mkdir(parents=True, exist_ok=True)


## Spectrogram generator function

In [4]:
def spectrogram_db(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    return 10 * np.log10(Sxx + EPS)


def create_spectrogram(sig, fs, db_range):
    db_min, db_max = db_range
    Sxx_db = spectrogram_db(sig, fs)
    Sxx_db = np.clip(Sxx_db, db_min, db_max)

    return 1.0 - (Sxx_db - db_min) / (db_max - db_min)

In [ ]:
def repo_relative(path: Path) -> str:
    path = Path(path).resolve()
    return str(path.relative_to(REPO_ROOT)) if path.is_relative_to(REPO_ROOT) else str(path)

dataset_manifests = {}
split_paths = {}
required_columns = {"source_dataset", "npz_path", "split", "label"}

for dataset_name in DATASETS_TO_PROCESS:
    config = DATASET_CONFIGS[dataset_name]
    manifest = pd.read_csv(config["manifest_path"])
    manifest = manifest[manifest["source_dataset"] == dataset_name].copy()
    missing_columns = required_columns - set(manifest.columns)
    if missing_columns:
        raise ValueError(f"{dataset_name} manifest is missing required columns: {sorted(missing_columns)}")
    dataset_manifests[dataset_name] = manifest

    for split in sorted(manifest["split"].unique()):
        split_labels = sorted(manifest.loc[manifest["split"] == split, "label"].unique())
        for label in split_labels:
            rows = manifest[(manifest["split"] == split) & (manifest["label"] == label)]
            split_paths[(dataset_name, split, label)] = [REPO_ROOT / rel_path for rel_path in rows["npz_path"]]

for (dataset_name, split, label), paths in split_paths.items():
    print(f"{dataset_name}/{split}/{label}: {len(paths)}")


def estimate_dataset_db_range(dataset_name, manifest):
    config = DATASET_CONFIGS[dataset_name]
    train_rows = manifest[manifest["split"] == "train"]
    if train_rows.empty:
        raise ValueError(f"{dataset_name} has no training rows for dB range estimation")

    if len(train_rows) > DB_RANGE_SAMPLE_LIMIT:
        train_rows = train_rows.sample(n=DB_RANGE_SAMPLE_LIMIT, random_state=SEED)

    db_values = []
    for rel_path in tqdm(train_rows["npz_path"], desc=f"Estimating {dataset_name} dB range"):
        data = np.load(REPO_ROOT / rel_path)
        fs = float(data["fs"]) if "fs" in data.files else config["sample_rate_hz"]
        for axis in ("X", "Y", "Z"):
            db_values.append(spectrogram_db(data[axis], fs).ravel())

    db_values = np.concatenate(db_values)
    db_values = db_values[np.isfinite(db_values)]
    db_min, db_max = np.percentile(db_values, DB_PERCENTILES)
    if db_max <= db_min:
        raise ValueError(f"Invalid {dataset_name} dB range: {(db_min, db_max)}")
    return float(db_min), float(db_max)


DATASET_DB_RANGES = {
    dataset_name: estimate_dataset_db_range(dataset_name, manifest)
    for dataset_name, manifest in dataset_manifests.items()
}

for dataset_name, db_range in DATASET_DB_RANGES.items():
    print(f"{dataset_name} dB range ({DB_PERCENTILES[0]}-{DB_PERCENTILES[1]} percentile): {db_range}")


turning/test/chatter: 27
turning/test/no_chatter: 48
turning/train/no_chatter: 472
turning/validation/chatter: 34
turning/validation/no_chatter: 70
cnc_machining/test/anomaly: 481
cnc_machining/test/nominal: 5144
cnc_machining/train/nominal: 23698
cnc_machining/validation/anomaly: 612
cnc_machining/validation/nominal: 4975


Estimating turning dB range: 100%|██████████| 472/472 [00:40<00:00, 11.73it/s]


# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [ ]:
def process_npz(dataset_name, npz_path, split_folder, target_label_folder):
    data = np.load(npz_path)
    config = DATASET_CONFIGS[dataset_name]

    x = data["X"]
    y = data["Y"]
    z = data["Z"]

    label = str(data["label"])
    # label_folder wird nicht mehr aus label abgeleitet, sondern kommt vom Caller
    fs = float(data["fs"]) if "fs" in data.files else config["sample_rate_hz"]
    db_range = DATASET_DB_RANGES[dataset_name]

    specs = [
        create_spectrogram(x, fs, db_range),
        create_spectrogram(y, fs, db_range),
        create_spectrogram(z, fs, db_range)
    ]

    rgb = np.stack(specs, axis=-1)
    rgb = np.clip(rgb, 0, 1)
    rgb = np.flipud(rgb)

    img = Image.fromarray((rgb * 255).astype(np.uint8))
    img = img.resize(TARGET_IMG_SIZE)

    experiment = npz_path.parent.parent.name
    out_name = f"{dataset_name}_{experiment}_{npz_path.stem}_{label}.png"

    out_path = config["output_root"] / split_folder / target_label_folder / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path)
    return out_path

In [ ]:
generated_rows = []

for (dataset_name, split_folder, label_folder), npz_files in split_paths.items():
    desc = f"Processing {dataset_name}/{split_folder}/{label_folder}"
    for npz_file in tqdm(npz_files, desc=desc):
        try:
            out_path = process_npz(dataset_name, npz_file, split_folder, label_folder)
            generated_rows.append({
                "source_dataset": dataset_name,
                "npz_path": repo_relative(npz_file),
                "image_path": repo_relative(out_path),
            })
        except Exception as e:
            print(f"Error in {npz_file}: {e}")

generated_images = pd.DataFrame(generated_rows)
if not generated_images.empty:
    for dataset_name, manifest in dataset_manifests.items():
        config = DATASET_CONFIGS[dataset_name]
        dataset_images = generated_images[generated_images["source_dataset"] == dataset_name]
        updated = manifest.drop(columns=["image_path"], errors="ignore").merge(
            dataset_images[["npz_path", "image_path"]], on="npz_path", how="left"
        )
        updated.to_csv(config["manifest_path"], index=False)
        print(f"Wrote image paths to {config['manifest_path']}")



Processing turning/test/chatter:   0%|          | 0/27 [00:00<?, ?it/s]


Processing turning/test/chatter:   4%|▎         | 1/27 [00:00<00:03,  7.08it/s]


Processing turning/test/chatter:  11%|█         | 3/27 [00:00<00:02,  9.20it/s]


Processing turning/test/chatter:  19%|█▊        | 5/27 [00:00<00:02,  9.65it/s]


Processing turning/test/chatter:  22%|██▏       | 6/27 [00:00<00:02,  9.72it/s]

...


Processing cnc_machining/validation/nominal:  99%|█████████▉| 4948/4975 [00:20<00:00, 236.20it/s]


Processing cnc_machining/validation/nominal: 100%|█████████▉| 4974/4975 [00:20<00:00, 242.35it/s]


Processing cnc_machining/validation/nominal: 100%|██████████| 4975/4975 [00:20<00:00, 242.31it/s]

Wrote image paths to ../reports/manifests/turning_split_seed42.csv


Wrote image paths to ../reports/manifests/cnc_machining_split_seed42.csv
